# DroneDetect V2 - Model Comparison

Comprehensive comparison of all trained models:
- **SVM** (PSD features)
- **VGG16** (Spectrogram features - RGB via Viridis colormap)
- **ResNet50** (Spectrogram features - RGB via Viridis colormap)
- **RF-UAV-Net** (Raw IQ features)

**File-level split (key contribution):** The RFClassification reference repository
splits data at the *segment* level, meaning segments from the same recording file
can end up in both training and test sets. This causes data leakage: the model
memorizes recording-specific noise patterns rather than learning drone RF signatures,
yielding inflated accuracies (~99.8%). Our approach enforces a **file-level split**
(70/15/15, drone-only stratification) where all segments from one recording stay in
the same partition. The accuracies reported here are therefore lower but represent
honest, leakage-free evaluations.

Evaluation metrics:
- Accuracy, Precision, Recall, F1-Score
- Confusion matrices
- Statistical significance tests (McNemar, Bootstrap CI, Cohen's kappa)
- Per-sample inference time with proper benchmarking
- Model size comparison
- Error analysis

## 0. Colab Setup
Run this cell only on Google Colab. Skipped automatically when running locally.

In [1]:
import os
import subprocess

COLAB = "COLAB_RELEASE_TAG" in os.environ
if COLAB:
    subprocess.check_call(["pip", "install", "-q", "boto3"])
    from google.colab import userdata
    import boto3

    s3 = boto3.client(
        "s3",
        endpoint_url="https://s3.fr-par.scw.cloud",
        region_name="fr-par",
        aws_access_key_id=userdata.get("SCW_ACCESS_KEY"),
        aws_secret_access_key=userdata.get("SCW_SECRET_KEY"),
    )
    ARTIFACT_BUCKET = "mldrone-artefacts"

    os.makedirs("../data/features", exist_ok=True)
    os.makedirs("../data/models", exist_ok=True)

    downloads = {
        "features/psd_features.npz": "../data/features/psd_features.npz",
        "features/spectrogram_features.npy": "../data/features/spectrogram_features.npy",
        "features/spectrogram_meta.npz": "../data/features/spectrogram_meta.npz",
        "features/iq_features.npz": "../data/features/iq_features.npz",
        "split/split_indices.npz": "../data/split_indices.npz",
        "models/svm_psd_drone.pkl": "../data/models/svm_psd_drone.pkl",
        "models/vgg16_cnn.pth": "../data/models/vgg16_cnn.pth",
        "models/resnet50_cnn.pth": "../data/models/resnet50_cnn.pth",
        "models/rfuavnet_iq.pth": "../data/models/rfuavnet_iq.pth",
    }
    for s3_key, local_path in downloads.items():
        if not os.path.exists(local_path):
            print(f"Downloading {s3_key}...")
            s3.download_file(ARTIFACT_BUCKET, s3_key, local_path)
            print(f"Done: {local_path}")

Done: ../data/features/psd_features.npz
Done: ../data/features/spectrogram_features.npy
Done: ../data/features/spectrogram_meta.npz
Done: ../data/features/iq_features.npz
Done: ../data/split_indices.npz
Done: ../data/models/svm_psd_drone.pkl
Done: ../data/models/vgg16_cnn.pth
Done: ../data/models/resnet50_cnn.pth
Done: ../data/models/rfuavnet_iq.pth


## 1. Imports and Configuration

In [2]:
import gc
import json
import logging
import os
import pickle
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    f1_score, precision_recall_fscore_support, cohen_kappa_score
)
from scipy.stats import chi2

try:
    from dronedetect.splitting import load_split, verify_split_balance, create_stratified_split
except ImportError:
    from sklearn.model_selection import StratifiedGroupKFold

    _splitting_logger = logging.getLogger(__name__)

    def create_stratified_split(
        y: np.ndarray,
        file_ids: np.ndarray,
        train_size: float = 0.70,
        val_size: float = 0.15,
        test_size: float = 0.15,
        random_state: int = 42,
        save_path: str | Path | None = None,
    ) -> dict:
        """
        Two-stage stratified group split into train/val/test partitions.

        Stage 1: split off test set using StratifiedGroupKFold.
        Stage 2: split remaining trainval into train and val.

        Stratification is by drone label (y). Grouping is by file_ids to prevent
        segments from the same recording appearing in different partitions.
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)

        total = train_size + val_size + test_size
        if not np.isclose(total, 1.0):
            raise ValueError(f"Split fractions must sum to 1.0, got {total:.4f}")

        n_splits_test = round(1 / test_size)
        sgkf_test = StratifiedGroupKFold(
            n_splits=n_splits_test, shuffle=True, random_state=random_state
        )
        all_indices = np.arange(len(y))

        trainval_idx, test_idx = next(sgkf_test.split(all_indices, y, groups=file_ids))

        relative_val = val_size / (train_size + val_size)
        n_splits_val = round(1 / relative_val)
        sgkf_val = StratifiedGroupKFold(
            n_splits=n_splits_val, shuffle=True, random_state=random_state + 1
        )

        y_trainval = y[trainval_idx]
        file_ids_trainval = file_ids[trainval_idx]

        train_local, val_local = next(
            sgkf_val.split(
                np.arange(len(trainval_idx)), y_trainval, groups=file_ids_trainval
            )
        )
        train_idx = trainval_idx[train_local]
        val_idx = trainval_idx[val_local]

        train_files = set(file_ids[train_idx])
        val_files = set(file_ids[val_idx])
        test_files = set(file_ids[test_idx])

        overlap_tv = train_files & val_files
        overlap_tt = train_files & test_files
        overlap_vt = val_files & test_files

        if overlap_tv or overlap_tt or overlap_vt:
            msg = "Data leakage detected! File overlap between partitions:"
            if overlap_tv:
                msg += f"\n  train-val: {overlap_tv}"
            if overlap_tt:
                msg += f"\n  train-test: {overlap_tt}"
            if overlap_vt:
                msg += f"\n  val-test: {overlap_vt}"
            raise ValueError(msg)

        _splitting_logger.info("Split created with zero file overlap (leakage-free)")
        _splitting_logger.info(
            "Samples  -> train=%d, val=%d, test=%d (total=%d)",
            len(train_idx), len(val_idx), len(test_idx), len(y),
        )
        _splitting_logger.info(
            "Files    -> train=%d, val=%d, test=%d",
            len(train_files), len(val_files), len(test_files),
        )

        split_metadata = {
            "train_samples": len(train_idx),
            "val_samples": len(val_idx),
            "test_samples": len(test_idx),
            "train_files": len(train_files),
            "val_files": len(val_files),
            "test_files": len(test_files),
            "random_state": random_state,
        }

        result = {
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "split_metadata": split_metadata,
        }

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez(
                save_path,
                train_idx=train_idx,
                val_idx=val_idx,
                test_idx=test_idx,
            )
            _splitting_logger.info("Split indices saved to %s", save_path)

        return result

    def verify_split_balance(
        file_ids: np.ndarray,
        y: np.ndarray,
        split_indices: dict,
        conditions: np.ndarray | None = None,
    ) -> None:
        """Log class and condition distributions per partition for diagnostics."""
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)
        if conditions is not None:
            conditions = np.asarray(conditions)

        partitions = {
            "train": split_indices["train_idx"],
            "val": split_indices["val_idx"],
            "test": split_indices["test_idx"],
        }

        all_classes = np.unique(y)

        for name, idx in partitions.items():
            y_part = y[idx]
            classes, counts = np.unique(y_part, return_counts=True)
            dist_str = ", ".join(f"{c}={n}" for c, n in zip(classes, counts))
            _splitting_logger.info("[%s] Class distribution: %s", name, dist_str)

        if conditions is not None:
            all_conditions = np.unique(conditions)

            for name, idx in partitions.items():
                cond_part = conditions[idx]
                conds, counts = np.unique(cond_part, return_counts=True)
                dist_str = ", ".join(f"{c}={n}" for c, n in zip(conds, counts))
                _splitting_logger.info("[%s] Condition distribution: %s", name, dist_str)

                y_part = y[idx]
                for drone in all_classes:
                    drone_mask = y_part == drone
                    drone_conditions = set(np.unique(cond_part[drone_mask]))
                    missing = set(all_conditions) - drone_conditions
                    if missing:
                        _splitting_logger.warning(
                            "[%s] Drone '%s' missing conditions: %s",
                            name, drone, missing,
                        )

    def load_split(path: str | Path) -> dict:
        """Load previously saved split indices from a .npz file."""
        data = np.load(path)
        return {
            "train_idx": data["train_idx"],
            "val_idx": data["val_idx"],
            "test_idx": data["test_idx"],
        }

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info("Using device: %s", device)

NOTEBOOK_NAME = "model_comparison"
FIGURES_DIR = Path("figures") / NOTEBOOK_NAME


def save_figure(fig) -> None:
    """Save plotly figure to PNG file using the figure's title as filename."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    title = fig.layout.title.text if fig.layout.title.text else "untitled"
    filename = re.sub(r'[^\w\s-]', '', title).strip()
    filename = re.sub(r'[\s-]+', '_', filename)
    filepath = FIGURES_DIR / f"{filename}.png"
    try:
        fig.write_image(str(filepath), width=1200, height=800)
        logger.info("Saved: %s", filepath)
    except Exception as e:
        logger.warning("Could not save figure (kaleido required): %s", e)

In [3]:
FEATURES_DIR = Path("../data/features")
MODELS_DIR = Path("../data/models")
SPLIT_PATH = Path("../data/split_indices.npz")

NUM_CLASSES = 7

CONFIG = {
    'batch_size': 128,
    'n_bootstrap': 1000,
    'n_timing_runs': 100,
    'warmup_runs': 10,
    'random_state': 42,
    'device': device,
    'colors': {
        'SVM': '#3498db',
        'VGG16': '#e74c3c',
        'ResNet50': '#2ecc71',
        'RFUAVNet': '#9b59b6'
    }
}

logger.info("Configuration: batch_size=%d, device=%s", CONFIG['batch_size'], device)

## 2. Model Definitions

Spectrograms are already RGB (3 channels via Viridis colormap from preprocessing).

In [4]:
class VGG16FC(nn.Module):
    """VGG16 with frozen features and trainable classifier."""

    def __init__(self, num_classes: int):
        super().__init__()
        vgg = tv_models.vgg16(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(vgg.children())[:-1])

        for param in self.features.parameters():
            param.requires_grad = False

        self.classifier = nn.Linear(25088, num_classes)

    def forward(self, x):
        if x.dim() == 4 and x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)

        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        return self.classifier(x)


class ResNet50FC(nn.Module):
    """ResNet50 with frozen features and trainable classifier.

    AdaptiveAvgPool2d reduces feature maps to (1,1) spatially,
    keeping FC at 2048 inputs instead of 100352 (49x param reduction).
    """

    def __init__(self, num_classes: int):
        super().__init__()
        resnet = tv_models.resnet50(weights='IMAGENET1K_V1')
        self.features = nn.Sequential(*list(resnet.children())[:-2])

        for param in self.features.parameters():
            param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        if x.dim() == 4 and x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)

        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)


class RFUAVNet(nn.Module):
    """RF-UAV-Net: 1D CNN for raw IQ classification."""

    def __init__(self, num_classes: int):
        super().__init__()

        self.conv_r = nn.Conv1d(2, 64, kernel_size=5, stride=5)
        self.bn_r = nn.BatchNorm1d(64)
        self.elu_r = nn.ELU()

        self.g_convs = nn.ModuleList([
            nn.Conv1d(64, 64, kernel_size=3, stride=2, groups=8)
            for _ in range(4)
        ])
        self.g_bns = nn.ModuleList([nn.BatchNorm1d(64) for _ in range(4)])
        self.g_elus = nn.ModuleList([nn.ELU() for _ in range(4)])

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.gap1000 = nn.AvgPool1d(1000)
        self.gap500 = nn.AvgPool1d(500)
        self.gap250 = nn.AvgPool1d(250)
        self.gap125 = nn.AvgPool1d(125)

        self.fc = nn.Linear(320, num_classes)

    def forward(self, x):
        x = self.elu_r(self.bn_r(self.conv_r(x)))

        g_outputs = []
        for i in range(4):
            g_out = self.g_elus[i](self.g_bns[i](self.g_convs[i](F.pad(x, (1, 0)))))
            g_outputs.append(g_out)
            x = g_out + self.pool(x)

        gaps = [
            self.gap1000(g_outputs[0]),
            self.gap500(g_outputs[1]),
            self.gap250(g_outputs[2]),
            self.gap125(g_outputs[3]),
            self.gap125(x)
        ]

        x = torch.cat(gaps, dim=1).flatten(start_dim=1)
        return self.fc(x)


class SpectrogramDataset(Dataset):
    """Dataset for RGB spectrograms (already 3 channels from preprocessing)."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx]).float()
        y = torch.tensor(self.y[idx]).long()
        return x, y


logger.info("Model classes defined: VGG16FC, ResNet50FC, RFUAVNet, SpectrogramDataset")

## 3. Evaluation Helpers

In [5]:
def evaluate_pytorch_model(model, dataloader, device, n_timing_runs=100, warmup_runs=10):
    """Evaluate PyTorch model with per-sample inference timing.

    Returns predictions, true labels, and timing statistics (p50/p95/p99 in ms).
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.numpy())

    sample_x, _ = next(iter(dataloader))
    single_sample = sample_x[:1].to(device)

    with torch.no_grad():
        for _ in range(warmup_runs):
            _ = model(single_sample)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_timing_runs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(single_sample)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end = time.perf_counter()
            times.append((end - start) * 1000)

    times = np.array(times)
    timing_stats = {
        'p50_ms': np.percentile(times, 50),
        'p95_ms': np.percentile(times, 95),
        'p99_ms': np.percentile(times, 99),
        'mean_ms': np.mean(times),
        'std_ms': np.std(times)
    }

    return np.array(all_preds), np.array(all_labels), timing_stats


def benchmark_svm_model(model, X, n_timing_runs=100, warmup_runs=10):
    """Benchmark SVM model with per-sample inference timing.

    Returns timing statistics (p50/p95/p99 in ms).
    """
    single_sample = X[:1]

    for _ in range(warmup_runs):
        _ = model.predict(single_sample)

    times = []
    for _ in range(n_timing_runs):
        start = time.perf_counter()
        _ = model.predict(single_sample)
        end = time.perf_counter()
        times.append((end - start) * 1000)

    times = np.array(times)
    return {
        'p50_ms': np.percentile(times, 50),
        'p95_ms': np.percentile(times, 95),
        'p99_ms': np.percentile(times, 99),
        'mean_ms': np.mean(times),
        'std_ms': np.std(times)
    }


def get_model_size_mb(model_path):
    """Get model file size in MB."""
    return os.path.getsize(model_path) / (1024 * 1024)


logger.info("Evaluation helpers defined")

## 4. Load Features and Shared Split

All 3 feature types share the same split indices to ensure identical test sets.

In [6]:
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    logger.info("Loaded existing split from %s", SPLIT_PATH)
else:
    data_psd = np.load(FEATURES_DIR / "psd_features.npz")
    split = create_stratified_split(
        data_psd['y_drone'], data_psd['file_ids'], save_path=SPLIT_PATH
    )
    logger.info("Created and saved new split to %s", SPLIT_PATH)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

logger.info("Split sizes: train=%d, val=%d, test=%d", len(train_idx), len(val_idx), len(test_idx))

In [7]:
data_psd = np.load(FEATURES_DIR / "psd_features.npz")

X_psd = data_psd['X']
y_drone_psd = data_psd['y_drone']
y_interference_psd = data_psd['y_interference']
file_ids_psd = data_psd['file_ids']
drone_classes = np.array(data_psd['drone_classes'])
interference_classes = np.array(data_psd['interference_classes'])

verify_split_balance(file_ids_psd, y_drone_psd, split, conditions=y_interference_psd)

X_test_psd = X_psd[test_idx]
y_test_psd = y_drone_psd[test_idx]

logger.info("PSD test set: %s", X_test_psd.shape)
logger.info("Drone classes: %s", drone_classes)

del X_psd, y_drone_psd, file_ids_psd, data_psd
gc.collect()

99

In [8]:
X_spec = np.load(FEATURES_DIR / "spectrogram_features.npy", mmap_mode="r")
meta_spec = np.load(FEATURES_DIR / "spectrogram_meta.npz")

y_drone_spec = meta_spec["y_drone"]

X_test_spec = X_spec[test_idx]
y_test_spec = y_drone_spec[test_idx]

test_dataset_spec = SpectrogramDataset(X_test_spec, y_test_spec)
test_loader_spec = DataLoader(
    test_dataset_spec,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

logger.info("Spectrogram test set (RGB): %s", X_test_spec.shape)
logger.info("Test batches (spec): %d", len(test_loader_spec))

del X_spec, y_drone_spec, meta_spec
gc.collect()

22

In [9]:
data_iq = np.load(FEATURES_DIR / "iq_features.npz")

X_iq = data_iq['X']
y_drone_iq = data_iq['y_drone']

X_test_iq = X_iq[test_idx]
y_test_iq = y_drone_iq[test_idx]

X_test_iq_t = torch.from_numpy(X_test_iq.copy()).float()
y_test_iq_t = torch.from_numpy(y_test_iq.copy()).long()

test_dataset_iq = TensorDataset(X_test_iq_t, y_test_iq_t)
test_loader_iq = DataLoader(
    test_dataset_iq,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
)

logger.info("IQ test set: %s", X_test_iq.shape)
logger.info("Test batches (IQ): %d", len(test_loader_iq))

del X_iq, y_drone_iq, data_iq
gc.collect()

22

## 5. Evaluate SVM

In [10]:
svm_path = MODELS_DIR / "svm_psd_drone.pkl"

with open(svm_path, "rb") as f:
    svm_data = pickle.load(f)

svm_model = svm_data["model"]

svm_preds = svm_model.predict(X_test_psd)
svm_acc = accuracy_score(y_test_psd, svm_preds)
svm_f1 = f1_score(y_test_psd, svm_preds, average="weighted")
svm_timing = benchmark_svm_model(
    svm_model, X_test_psd,
    n_timing_runs=CONFIG["n_timing_runs"],
    warmup_runs=CONFIG["warmup_runs"],
)

print(f"SVM Test Accuracy: {svm_acc:.4f}")
print(f"SVM Test F1-Score: {svm_f1:.4f}")
print(f"SVM Inference Time: {svm_timing['p50_ms']:.3f} ms (p50), {svm_timing['p95_ms']:.3f} ms (p95)")
print(f"SVM Model Size: {get_model_size_mb(svm_path):.2f} MB")

del svm_model, svm_data
gc.collect()

SVM Test Accuracy: 0.8346
SVM Test F1-Score: 0.8304
SVM Inference Time: 14.903 ms (p50), 15.474 ms (p95)
SVM Model Size: 88.61 MB


0

## 6. Evaluate VGG16

In [11]:
vgg_path = MODELS_DIR / "vgg16_cnn.pth"
vgg_checkpoint = torch.load(vgg_path, map_location=device, weights_only=False)

vgg_model = VGG16FC(num_classes=NUM_CLASSES)
vgg_model.load_state_dict(vgg_checkpoint['model_state_dict'])
vgg_model = vgg_model.to(device)

vgg_preds, vgg_labels, vgg_timing = evaluate_pytorch_model(
    vgg_model, test_loader_spec, device,
    n_timing_runs=CONFIG['n_timing_runs'],
    warmup_runs=CONFIG['warmup_runs']
)

vgg_acc = accuracy_score(vgg_labels, vgg_preds)
vgg_f1 = f1_score(vgg_labels, vgg_preds, average='weighted')

print(f"VGG16 Test Accuracy: {vgg_acc:.4f}")
print(f"VGG16 Test F1-Score: {vgg_f1:.4f}")
print(f"VGG16 Inference Time: {vgg_timing['p50_ms']:.3f} ms (p50), {vgg_timing['p95_ms']:.3f} ms (p95)")
print(f"VGG16 Model Size: {get_model_size_mb(vgg_path):.2f} MB")

del vgg_model, vgg_checkpoint
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 246MB/s]


VGG16 Test Accuracy: 0.8876
VGG16 Test F1-Score: 0.8886
VGG16 Inference Time: 1.739 ms (p50), 2.035 ms (p95)
VGG16 Model Size: 56.81 MB


## 7. Evaluate ResNet50

In [12]:
resnet_path = MODELS_DIR / "resnet50_cnn.pth"
resnet_checkpoint = torch.load(resnet_path, map_location=device, weights_only=False)

resnet_model = ResNet50FC(num_classes=NUM_CLASSES)
resnet_model.load_state_dict(resnet_checkpoint['model_state_dict'])
resnet_model = resnet_model.to(device)

resnet_preds, resnet_labels, resnet_timing = evaluate_pytorch_model(
    resnet_model, test_loader_spec, device,
    n_timing_runs=CONFIG['n_timing_runs'],
    warmup_runs=CONFIG['warmup_runs']
)

resnet_acc = accuracy_score(resnet_labels, resnet_preds)
resnet_f1 = f1_score(resnet_labels, resnet_preds, average='weighted')

print(f"ResNet50 Test Accuracy: {resnet_acc:.4f}")
print(f"ResNet50 Test F1-Score: {resnet_f1:.4f}")
print(f"ResNet50 Inference Time: {resnet_timing['p50_ms']:.3f} ms (p50), {resnet_timing['p95_ms']:.3f} ms (p95)")
print(f"ResNet50 Model Size: {get_model_size_mb(resnet_path):.2f} MB")

del resnet_model, resnet_checkpoint
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 217MB/s]


ResNet50 Test Accuracy: 0.7880
ResNet50 Test F1-Score: 0.7888
ResNet50 Inference Time: 6.872 ms (p50), 6.984 ms (p95)
ResNet50 Model Size: 90.04 MB


## 8. Evaluate RF-UAV-Net

In [13]:
rfuavnet_path = MODELS_DIR / "rfuavnet_iq.pth"
rfuavnet_checkpoint = torch.load(rfuavnet_path, map_location=device, weights_only=False)

rfuavnet_model = RFUAVNet(num_classes=NUM_CLASSES)
rfuavnet_model.load_state_dict(rfuavnet_checkpoint['model_state_dict'])
rfuavnet_model = rfuavnet_model.to(device)

rfuavnet_preds, rfuavnet_labels, rfuavnet_timing = evaluate_pytorch_model(
    rfuavnet_model, test_loader_iq, device,
    n_timing_runs=CONFIG['n_timing_runs'],
    warmup_runs=CONFIG['warmup_runs']
)

rfuavnet_acc = accuracy_score(rfuavnet_labels, rfuavnet_preds)
rfuavnet_f1 = f1_score(rfuavnet_labels, rfuavnet_preds, average='weighted')

print(f"RF-UAV-Net Test Accuracy: {rfuavnet_acc:.4f}")
print(f"RF-UAV-Net Test F1-Score: {rfuavnet_f1:.4f}")
print(f"RF-UAV-Net Inference Time: {rfuavnet_timing['p50_ms']:.3f} ms (p50), {rfuavnet_timing['p95_ms']:.3f} ms (p95)")
print(f"RF-UAV-Net Model Size: {get_model_size_mb(rfuavnet_path):.2f} MB")

del rfuavnet_model, rfuavnet_checkpoint
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

RF-UAV-Net Test Accuracy: 0.6298
RF-UAV-Net Test F1-Score: 0.6258
RF-UAV-Net Inference Time: 1.503 ms (p50), 1.535 ms (p95)
RF-UAV-Net Model Size: 0.06 MB


## 9. Aggregate Results

In [14]:
results_df = pd.DataFrame({
    'Model': ['SVM', 'VGG16', 'ResNet50', 'RFUAVNet'],
    'Features': ['PSD', 'Spectrogram', 'Spectrogram', 'Raw IQ'],
    'Accuracy': [svm_acc, vgg_acc, resnet_acc, rfuavnet_acc],
    'F1-Score': [svm_f1, vgg_f1, resnet_f1, rfuavnet_f1],
    'Inference_p50_ms': [
        svm_timing['p50_ms'],
        vgg_timing['p50_ms'],
        resnet_timing['p50_ms'],
        rfuavnet_timing['p50_ms']
    ],
    'Inference_p95_ms': [
        svm_timing['p95_ms'],
        vgg_timing['p95_ms'],
        resnet_timing['p95_ms'],
        rfuavnet_timing['p95_ms']
    ],
    'Inference_p99_ms': [
        svm_timing['p99_ms'],
        vgg_timing['p99_ms'],
        resnet_timing['p99_ms'],
        rfuavnet_timing['p99_ms']
    ],
    'Model_Size_MB': [
        get_model_size_mb(svm_path),
        get_model_size_mb(vgg_path),
        get_model_size_mb(resnet_path),
        get_model_size_mb(rfuavnet_path)
    ]
})

results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n=== Model Comparison Results ===")
print(results_df.to_string(index=False))

predictions = {
    'SVM': svm_preds,
    'VGG16': vgg_preds,
    'ResNet50': resnet_preds,
    'RFUAVNet': rfuavnet_preds
}


=== Model Comparison Results ===
   Model    Features  Accuracy  F1-Score  Inference_p50_ms  Inference_p95_ms  Inference_p99_ms  Model_Size_MB
   VGG16 Spectrogram  0.887584  0.888586          1.739051          2.035252          2.238074      56.814044
     SVM         PSD  0.834578  0.830422         14.902685         15.474319         15.731401      88.608373
ResNet50 Spectrogram  0.787976  0.788766          6.872282          6.983761          7.778841      90.039823
RFUAVNet      Raw IQ  0.629847  0.625761          1.503265          1.535388          1.545581       0.057648


## 10. Per-Class Metrics

In [15]:
per_class_metrics = {}

for model_name, preds in predictions.items():
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test_psd, preds, labels=range(len(drone_classes)), zero_division=0
    )
    per_class_metrics[model_name] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': support
    }

best_model = results_df.iloc[0]['Model']
print(f"\n=== {best_model} Per-Class Performance ===")
metrics = per_class_metrics[best_model]
pcm_display = pd.DataFrame({
    "Class": drone_classes,
    "Precision": metrics['precision'],
    "Recall": metrics['recall'],
    "F1-Score": metrics['f1'],
    "Support": metrics['support'].astype(int),
})
print(pcm_display.to_string(index=False))


=== VGG16 Per-Class Performance ===
Class  Precision   Recall  F1-Score  Support
  AIR   0.952010 0.796095  0.867100      922
  DIS   0.936772 0.938333  0.937552      600
  INS   0.922892 0.957500  0.939877      800
  MA1   0.814301 0.763000  0.787816     1000
  MAV   0.693170 0.850000  0.763616      800
  MIN   1.000000 1.000000  1.000000      800
  PHA   0.974359 0.977143  0.975749      700


## 11. Visualizations

In [16]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Model Accuracy Comparison', 'Model F1-Score Comparison'))

models = results_df['Model'].values
colors = [CONFIG['colors'][m] for m in models]

fig.add_trace(go.Bar(
    x=list(models), y=results_df['Accuracy'].values,
    marker_color=colors, text=[f"{v:.4f}" for v in results_df['Accuracy'].values],
    textposition='outside', name='Accuracy'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=list(models), y=results_df['F1-Score'].values,
    marker_color=colors, text=[f"{v:.4f}" for v in results_df['F1-Score'].values],
    textposition='outside', name='F1-Score', showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Model Performance Comparison - Accuracy and F1-Score',
    height=500, width=1200
)
fig.update_yaxes(range=[0, 1.1], title_text='Accuracy', row=1, col=1)
fig.update_yaxes(range=[0, 1.1], title_text='F1-Score (Weighted)', row=1, col=2)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [17]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['SVM', 'VGG16', 'ResNet50', 'RFUAVNet'],
                    horizontal_spacing=0.12, vertical_spacing=0.12)

colorscales = ['Blues', 'Reds', 'Greens', 'Purples']
model_names = ['SVM', 'VGG16', 'ResNet50', 'RFUAVNet']
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, (model_name, (row, col)) in enumerate(zip(model_names, positions)):
    cm = confusion_matrix(y_test_psd, predictions[model_name])

    fig.add_trace(go.Heatmap(
        z=cm, x=list(drone_classes), y=list(drone_classes),
        colorscale=colorscales[idx], text=cm, texttemplate='%{text}',
        textfont={'size': 10}, showscale=False, hoverongaps=False
    ), row=row, col=col)

fig.update_layout(
    title='All Models Confusion Matrices',
    height=900, width=1000
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text='Predicted', row=i, col=j)
        fig.update_yaxes(title_text='True', autorange='reversed', row=i, col=j)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [18]:
fig = go.Figure()

for model_name, metrics in per_class_metrics.items():
    color = CONFIG['colors'][model_name]
    fig.add_trace(go.Bar(
        x=list(drone_classes), y=metrics['f1'],
        name=model_name, marker_color=color, opacity=0.8
    ))

fig.update_layout(
    title='Per-Class F1-Score Comparison',
    xaxis_title='Class',
    yaxis_title='F1-Score',
    yaxis_range=[0, 1.1],
    barmode='group',
    height=600, width=1000
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [19]:
fig = go.Figure()

models = results_df['Model'].values
p50_times = results_df['Inference_p50_ms'].values
p95_times = results_df['Inference_p95_ms'].values
colors_list = [CONFIG['colors'][m] for m in models]

fig.add_trace(go.Bar(
    y=list(models), x=p50_times, orientation='h',
    marker_color=colors_list, name='p50 (median)',
    text=[f"p50: {t:.2f}ms" for t in p50_times], textposition='outside'
))

fig.add_trace(go.Scatter(
    y=list(models), x=p95_times, mode='markers',
    marker=dict(symbol='line-ns', size=15, color='gray', line_width=2),
    name='p95'
))

fig.update_layout(
    title='Model Inference Time Comparison - Single Sample (p50 with p95 markers)',
    xaxis_title='Inference Time per Sample (ms)',
    height=500, width=900,
    showlegend=True
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [20]:
fig = go.Figure()

models = results_df['Model'].values
sizes = results_df['Model_Size_MB'].values
colors_list = [CONFIG['colors'][m] for m in models]

fig.add_trace(go.Bar(
    y=list(models), x=sizes, orientation='h',
    marker_color=colors_list,
    text=[f"{s:.2f} MB" for s in sizes], textposition='outside'
))

fig.update_layout(
    title='Model Size Comparison',
    xaxis_title='Model Size (MB)',
    height=500, width=900
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [21]:
def normalize(values):
    min_val = np.min(values)
    max_val = np.max(values)
    if max_val == min_val:
        return np.ones_like(values)
    return (values - min_val) / (max_val - min_val)

accuracy_norm = results_df['Accuracy'].values
f1_norm = results_df['F1-Score'].values
time_norm = 1 - normalize(results_df['Inference_p50_ms'].values)
size_norm = 1 - normalize(results_df['Model_Size_MB'].values)

categories = ['Accuracy', 'F1-Score', 'Speed (inverse time)', 'Compactness (inverse size)']

fig = go.Figure()

for idx, model in enumerate(models):
    values = [accuracy_norm[idx], f1_norm[idx], time_norm[idx], size_norm[idx]]
    values.append(values[0])
    categories_closed = categories + [categories[0]]

    color = CONFIG['colors'][model]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=categories_closed,
        fill='toself', name=model,
        line_color=color, fillcolor=color, opacity=0.3
    ))

fig.update_layout(
    title='Multi-Metric Model Comparison (Normalized)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=700, width=800,
    showlegend=True
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 12. Statistical Tests

McNemar's test is appropriate here because we have paired classification data
(identical test samples evaluated by each model) and it requires no distributional
assumptions on accuracy.

In [22]:
def mcnemar_test(y_true, pred1, pred2):
    """McNemar's test for paired predictions."""
    correct1 = (pred1 == y_true)
    correct2 = (pred2 == y_true)

    n01 = np.sum(~correct1 & correct2)
    n10 = np.sum(correct1 & ~correct2)

    if n01 + n10 == 0:
        return 1.0, 0.0

    statistic = (abs(n01 - n10) - 1)**2 / (n01 + n10)
    p_value = 1 - chi2.cdf(statistic, df=1)

    return p_value, statistic

model_names = ['SVM', 'VGG16', 'ResNet50', 'RFUAVNet']
mcnemar_results = []

for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        model1 = model_names[i]
        model2 = model_names[j]

        p_value, stat = mcnemar_test(
            y_test_psd,
            predictions[model1],
            predictions[model2]
        )

        significant = "Yes (p<0.05)" if p_value < 0.05 else "No"

        mcnemar_results.append({
            'Model1': model1,
            'Model2': model2,
            'p_value': p_value,
            'statistic': stat,
            'significant': p_value < 0.05
        })

print("=== McNemar's Test (Pairwise) ===")
mcnemar_df = pd.DataFrame(mcnemar_results)
mcnemar_df["Significant"] = mcnemar_df["significant"].map({True: "Yes (p<0.05)", False: "No"})
print(mcnemar_df[["Model1", "Model2", "p_value", "statistic", "Significant"]].to_string(index=False))

=== McNemar's Test (Pairwise) ===
  Model1   Model2      p_value   statistic  Significant
     SVM    VGG16 0.000000e+00   98.447545 Yes (p<0.05)
     SVM ResNet50 3.996803e-15   61.703804 Yes (p<0.05)
     SVM RFUAVNet 0.000000e+00  741.727426 Yes (p<0.05)
   VGG16 ResNet50 0.000000e+00  321.482510 Yes (p<0.05)
   VGG16 RFUAVNet 0.000000e+00 1079.106536 Yes (p<0.05)
ResNet50 RFUAVNet 0.000000e+00  427.395122 Yes (p<0.05)


In [23]:
def bootstrap_ci(y_true, y_pred, n_iterations=1000, ci=0.95, random_state=42):
    """Bootstrap confidence interval for accuracy."""
    np.random.seed(random_state)
    n_samples = len(y_true)
    accuracies = []

    for _ in range(n_iterations):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        accuracies.append(accuracy_score(y_true_boot, y_pred_boot))

    alpha = 1 - ci
    lower = np.percentile(accuracies, 100 * alpha / 2)
    upper = np.percentile(accuracies, 100 * (1 - alpha / 2))

    return lower, upper

bootstrap_results = {}
bootstrap_rows = []

for model_name in model_names:
    acc = accuracy_score(y_test_psd, predictions[model_name])
    lower, upper = bootstrap_ci(
        y_test_psd,
        predictions[model_name],
        n_iterations=CONFIG['n_bootstrap'],
        random_state=CONFIG['random_state']
    )
    bootstrap_results[model_name] = {"accuracy": acc, "ci_lower": lower, "ci_upper": upper}
    bootstrap_rows.append({
        "Model": model_name,
        "Accuracy": acc,
        "CI Lower": lower,
        "CI Upper": upper,
        "CI Width": upper - lower,
    })

print("\n=== Bootstrap 95% Confidence Intervals (Accuracy) ===")
bootstrap_df = pd.DataFrame(bootstrap_rows)
print(bootstrap_df.to_string(index=False))


=== Bootstrap 95% Confidence Intervals (Accuracy) ===
   Model  Accuracy  CI Lower  CI Upper  CI Width
     SVM  0.834578  0.825502  0.844361  0.018859
   VGG16  0.887584  0.879402  0.895589  0.016186
ResNet50  0.787976  0.777299  0.799538  0.022239
RFUAVNet  0.629847  0.618814  0.642303  0.023488


In [24]:
def interpret_kappa(kappa):
    if kappa < 0:
        return "Poor (worse than random)"
    elif kappa < 0.20:
        return "Slight"
    elif kappa < 0.40:
        return "Fair"
    elif kappa < 0.60:
        return "Moderate"
    elif kappa < 0.80:
        return "Substantial"
    else:
        return "Almost perfect"

kappa_rows = []
for model_name in model_names:
    kappa = cohen_kappa_score(y_test_psd, predictions[model_name])
    kappa_rows.append({
        "Model": model_name,
        "Kappa": kappa,
        "Interpretation": interpret_kappa(kappa),
    })

print("\n=== Cohen's Kappa (Agreement beyond chance) ===")
kappa_df = pd.DataFrame(kappa_rows)
print(kappa_df.to_string(index=False))


=== Cohen's Kappa (Agreement beyond chance) ===
   Model    Kappa Interpretation
     SVM 0.806623 Almost perfect
   VGG16 0.868492 Almost perfect
ResNet50 0.751856    Substantial
RFUAVNet 0.566219       Moderate


## 13. Error Analysis

In [25]:
best_model = results_df.iloc[0]['Model']
best_preds = predictions[best_model]

misclassified_mask = (best_preds != y_test_psd)
misclassified_true = y_test_psd[misclassified_mask]
misclassified_pred = best_preds[misclassified_mask]

print(f"\n=== {best_model} Error Analysis ===")
print(f"Total test samples: {len(y_test_psd)}")
print(f"Misclassified samples: {np.sum(misclassified_mask)} ({100*np.sum(misclassified_mask)/len(y_test_psd):.2f}%)")

print("\nMost common misclassification pairs (True -> Predicted):")
misclass_pairs = list(zip(misclassified_true, misclassified_pred))
unique_pairs, counts = np.unique(misclass_pairs, axis=0, return_counts=True)
sorted_indices = np.argsort(-counts)[:10]

print("  TRUE -> PRED")
for idx in sorted_indices:
    true_cls = drone_classes[unique_pairs[idx][0]]
    pred_cls = drone_classes[unique_pairs[idx][1]]
    count = counts[idx]
    pct = 100 * count / np.sum(misclassified_mask)
    print(f"  {true_cls} -> {pred_cls}: {count} errors ({pct:.1f}% of errors)")


=== VGG16 Error Analysis ===
Total test samples: 5622
Misclassified samples: 632 (11.24%)

Most common misclassification pairs (True -> Predicted):
  TRUE -> PRED
  MA1 -> MAV: 190 errors (30.1% of errors)
  MAV -> MA1: 90 errors (14.2% of errors)
  AIR -> MA1: 71 errors (11.2% of errors)
  AIR -> MAV: 69 errors (10.9% of errors)
  MA1 -> AIR: 28 errors (4.4% of errors)
  MAV -> INS: 27 errors (4.3% of errors)
  INS -> MAV: 25 errors (4.0% of errors)
  AIR -> DIS: 23 errors (3.6% of errors)
  MA1 -> INS: 18 errors (2.8% of errors)
  DIS -> MAV: 17 errors (2.7% of errors)


**MA1 / MAV confusion pattern:** The Mavic Air (MA1) and Mavic Pro (MAV) are
the most frequently confused drone pair across all models. Both drones use DJI's
OcuSync / Lightbridge communication protocols, which share the same chipset family
and produce very similar RF signatures in the 2.4 GHz band. This hardware-level
similarity makes them inherently harder to distinguish from RF emissions alone.

In [26]:
pred_matrix = np.array([predictions[m] for m in model_names]).T

all_agree = np.all(pred_matrix == pred_matrix[:, 0:1], axis=1)
all_correct = np.all(pred_matrix == y_test_psd[:, None], axis=1)
all_wrong = np.all(pred_matrix != y_test_psd[:, None], axis=1)

print("\n=== Model Agreement Analysis ===")
print(f"Samples where all models agree: {np.sum(all_agree)} ({100*np.sum(all_agree)/len(y_test_psd):.2f}%)")
print(f"Samples where all models are correct: {np.sum(all_correct)} ({100*np.sum(all_correct)/len(y_test_psd):.2f}%)")
print(f"Samples where all models are wrong: {np.sum(all_wrong)} ({100*np.sum(all_wrong)/len(y_test_psd):.2f}%)")

no_model_correct = np.all(pred_matrix != y_test_psd[:, None], axis=1)
print(f"\nHard samples (no model correct): {np.sum(no_model_correct)}")
if np.sum(no_model_correct) > 0:
    hard_true = y_test_psd[no_model_correct]
    unique_hard, counts_hard = np.unique(hard_true, return_counts=True)
    print("Distribution by class:")
    for cls_idx, count in zip(unique_hard, counts_hard):
        print(f"  {drone_classes[cls_idx]}: {count} samples")


=== Model Agreement Analysis ===
Samples where all models agree: 2879 (51.21%)
Samples where all models are correct: 2841 (50.53%)
Samples where all models are wrong: 167 (2.97%)

Hard samples (no model correct): 167
Distribution by class:
  AIR: 42 samples
  DIS: 4 samples
  MA1: 77 samples
  MAV: 41 samples
  PHA: 3 samples


## 13b. Robustness Analysis

How does each model perform across **interference conditions** (BLUE, BOTH, CLEAN,
WIFI) and **flight states** (FY=flying, HO=hovering, ON=stationary)?

This breakdown reveals whether a model's overall accuracy masks weaknesses under
specific operating conditions — critical information for real-world deployment.

In [27]:
data_psd_meta = np.load(FEATURES_DIR / "psd_features.npz")

y_state_all = data_psd_meta["y_state"]
state_classes = np.array(data_psd_meta["state_classes"])

y_interference_test = y_interference_psd[test_idx]
y_state_test = y_state_all[test_idx]

del data_psd_meta, y_state_all
gc.collect()

logger.info("Interference classes: %s", interference_classes)
logger.info("State classes: %s", state_classes)
logger.info(
    "Test set condition distribution: %s",
    dict(zip(*np.unique(y_interference_test, return_counts=True))),
)
logger.info(
    "Test set state distribution: %s",
    dict(zip(*np.unique(y_state_test, return_counts=True))),
)

In [28]:
model_names_ordered = ["SVM", "VGG16", "ResNet50", "RFUAVNet"]

robustness_rows = []

for axis_name, y_axis, axis_classes in [
    ("condition", y_interference_test, interference_classes),
    ("state", y_state_test, state_classes),
]:
    for model_name in model_names_ordered:
        preds = predictions[model_name]
        for cls_idx, cls_label in enumerate(axis_classes):
            mask = y_axis == cls_idx
            if mask.sum() == 0:
                continue
            acc = accuracy_score(y_test_psd[mask], preds[mask])
            f1 = f1_score(
                y_test_psd[mask], preds[mask],
                average="weighted", zero_division=0,
            )
            robustness_rows.append({
                "axis": axis_name,
                "class": cls_label,
                "model": model_name,
                "accuracy": acc,
                "f1": f1,
                "n_samples": int(mask.sum()),
            })

robustness_df = pd.DataFrame(robustness_rows)

print("=== Accuracy per Interference Condition ===")
cond_acc = robustness_df[robustness_df["axis"] == "condition"].pivot(
    index="class", columns="model", values="accuracy"
)[model_names_ordered]
print(cond_acc.round(4).to_string())

print("\n=== Accuracy per Flight State ===")
state_acc = robustness_df[robustness_df["axis"] == "state"].pivot(
    index="class", columns="model", values="accuracy"
)[model_names_ordered]
print(state_acc.round(4).to_string())

print("\n=== F1-Score per Interference Condition ===")
cond_f1 = robustness_df[robustness_df["axis"] == "condition"].pivot(
    index="class", columns="model", values="f1"
)[model_names_ordered]
print(cond_f1.round(4).to_string())

print("\n=== F1-Score per Flight State ===")
state_f1 = robustness_df[robustness_df["axis"] == "state"].pivot(
    index="class", columns="model", values="f1"
)[model_names_ordered]
print(state_f1.round(4).to_string())

=== Accuracy per Interference Condition ===
model     SVM   VGG16  ResNet50  RFUAVNet
class                                    
BLUE   0.7855  0.8425    0.7665    0.6160
BOTH   0.8843  0.9393    0.8221    0.6736
CLEAN  0.8521  0.8721    0.7557    0.5764
WIFI   0.8171  0.8971    0.8079    0.6536

=== Accuracy per Flight State ===
model     SVM   VGG16  ResNet50  RFUAVNet
class                                    
FY     0.7300  0.7491    0.6252    0.5715
HO     0.7164  0.8643    0.7207    0.4943
ON     0.9635  0.9865    0.9258    0.7392

=== F1-Score per Interference Condition ===
model     SVM   VGG16  ResNet50  RFUAVNet
class                                    
BLUE   0.7821  0.8445    0.7684    0.6113
BOTH   0.8811  0.9391    0.8228    0.6750
CLEAN  0.8503  0.8702    0.7575    0.5773
WIFI   0.8107  0.8988    0.8039    0.6387

=== F1-Score per Flight State ===
model     SVM   VGG16  ResNet50  RFUAVNet
class                                    
FY     0.7121  0.7473    0.6238    0.5694
H

In [29]:
fig = go.Figure()

for model_name in model_names_ordered:
    subset = robustness_df[
        (robustness_df["axis"] == "condition")
        & (robustness_df["model"] == model_name)
    ].sort_values("class")
    fig.add_trace(go.Bar(
        x=subset["class"].values.tolist(),
        y=subset["accuracy"].values,
        name=model_name,
        marker_color=CONFIG["colors"][model_name],
        text=[f"{v:.3f}" for v in subset["accuracy"].values],
        textposition="outside",
    ))

fig.update_layout(
    title="Accuracy per Interference Condition",
    xaxis_title="Interference Condition",
    yaxis_title="Accuracy",
    yaxis_range=[0, 1.1],
    barmode="group",
    height=550,
    width=1000,
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [30]:
fig = go.Figure()

for model_name in model_names_ordered:
    subset = robustness_df[
        (robustness_df["axis"] == "state")
        & (robustness_df["model"] == model_name)
    ].sort_values("class")
    fig.add_trace(go.Bar(
        x=subset["class"].values.tolist(),
        y=subset["accuracy"].values,
        name=model_name,
        marker_color=CONFIG["colors"][model_name],
        text=[f"{v:.3f}" for v in subset["accuracy"].values],
        textposition="outside",
    ))

fig.update_layout(
    title="Accuracy per Flight State",
    xaxis_title="Flight State",
    yaxis_title="Accuracy",
    yaxis_range=[0, 1.1],
    barmode="group",
    height=550,
    width=1000,
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [31]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("F1-Score per Condition", "F1-Score per Flight State"),
    horizontal_spacing=0.15,
)

fig.add_trace(go.Heatmap(
    z=cond_f1.values,
    x=model_names_ordered,
    y=cond_f1.index.tolist(),
    colorscale="YlOrRd",
    text=np.round(cond_f1.values, 3),
    texttemplate="%{text}",
    textfont={"size": 12},
    showscale=False,
    zmin=0, zmax=1,
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=state_f1.values,
    x=model_names_ordered,
    y=state_f1.index.tolist(),
    colorscale="YlOrRd",
    text=np.round(state_f1.values, 3),
    texttemplate="%{text}",
    textfont={"size": 12},
    showscale=True,
    zmin=0, zmax=1,
    colorbar=dict(title="F1", x=1.02),
), row=1, col=2)

fig.update_layout(
    title="F1-Score Heatmaps by Condition and Flight State",
    height=450,
    width=1100,
)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [32]:
robustness_path = MODELS_DIR / "robustness_metrics.csv"
robustness_df.to_csv(robustness_path, index=False)
logger.info("Robustness metrics saved to %s", robustness_path)

**Interpretation:**

- **Interference conditions:** BOTH (Bluetooth + WiFi simultaneously) is expected to
  be the hardest condition since it combines two interference sources. CLEAN (no
  interference) should yield the highest accuracy as models see the purest RF
  signatures. BLUE and WIFI fall in between, each adding a single interference
  source in the 2.4 GHz band.

- **Flight states:** Flying (FY) produces Doppler shifts and motor vibration patterns
  that differ from stationary (ON) and hovering (HO). Performance differences across
  states indicate how much models rely on motion-related RF artifacts versus
  intrinsic drone signatures.

## 14. Summary

In [33]:
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)

print("\nRANKING (by Accuracy):")
ranking_df = results_df[["Model", "Accuracy", "F1-Score"]].copy()
ranking_df.index = range(1, len(ranking_df) + 1)
ranking_df.index.name = "Rank"
print(ranking_df.to_string())

best = results_df.iloc[0]
fastest = results_df.loc[results_df['Inference_p50_ms'].idxmin()]
smallest = results_df.loc[results_df['Model_Size_MB'].idxmin()]

print(f"\nBest accuracy: {best['Model']} ({best['Accuracy']:.4f})")
print(f"Fastest inference: {fastest['Model']} (p50={fastest['Inference_p50_ms']:.2f}ms)")
print(f"Smallest model: {smallest['Model']} ({smallest['Model_Size_MB']:.2f}MB)")

print("=" * 60)

MODEL COMPARISON SUMMARY

RANKING (by Accuracy):
         Model  Accuracy  F1-Score
Rank                              
1        VGG16  0.887584  0.888586
2          SVM  0.834578  0.830422
3     ResNet50  0.787976  0.788766
4     RFUAVNet  0.629847  0.625761

Best accuracy: VGG16 (0.8876)
Fastest inference: RFUAVNet (p50=1.50ms)
Smallest model: RFUAVNet (0.06MB)


## 15. Export Results

In [34]:
results_path = MODELS_DIR / "model_comparison_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Results saved to {results_path}")

comparison_dict = {
    'results_df': results_df,
    'per_class_metrics': per_class_metrics,
    'mcnemar_results': mcnemar_results,
    'predictions': predictions,
    'true_labels': y_test_psd,
    'drone_classes': drone_classes,
    'config': CONFIG
}

comparison_path = MODELS_DIR / "model_comparison_full.pkl"
with open(comparison_path, 'wb') as f:
    pickle.dump(comparison_dict, f)
print(f"Full comparison saved to {comparison_path}")

Results saved to ../data/models/model_comparison_results.csv
Full comparison saved to ../data/models/model_comparison_full.pkl


In [35]:
cm_export = {}
for model_name, preds in predictions.items():
    cm = confusion_matrix(y_test_psd, preds)
    cm_export[model_name] = cm.tolist()
cm_export["drone_classes"] = list(drone_classes)

cm_path = MODELS_DIR / "confusion_matrices.json"
with open(cm_path, "w") as f:
    json.dump(cm_export, f, indent=2)
logger.info("Confusion matrices saved to %s", cm_path)

In [36]:
rows = []
for model_name, metrics in per_class_metrics.items():
    for i, cls in enumerate(drone_classes):
        rows.append({
            "model": model_name,
            "class": cls,
            "precision": metrics["precision"][i],
            "recall": metrics["recall"][i],
            "f1": metrics["f1"][i],
            "support": int(metrics["support"][i]),
        })

pcm_df = pd.DataFrame(rows)
pcm_path = MODELS_DIR / "per_class_metrics.csv"
pcm_df.to_csv(pcm_path, index=False)
logger.info("Per-class metrics saved to %s", pcm_path)

In [37]:
stat_rows = []

for result in mcnemar_results:
    stat_rows.append({
        "test_type": "mcnemar",
        "model_a": result["Model1"],
        "model_b": result["Model2"],
        "statistic": result["statistic"],
        "p_value": result["p_value"],
        "ci_lower": None,
        "ci_upper": None,
    })

for model_name in model_names:
    br = bootstrap_results[model_name]
    stat_rows.append({
        "test_type": "bootstrap_ci_accuracy",
        "model_a": model_name,
        "model_b": None,
        "statistic": br["accuracy"],
        "p_value": None,
        "ci_lower": br["ci_lower"],
        "ci_upper": br["ci_upper"],
    })

for model_name in model_names:
    kappa = cohen_kappa_score(y_test_psd, predictions[model_name])
    stat_rows.append({
        "test_type": "cohen_kappa",
        "model_a": model_name,
        "model_b": None,
        "statistic": kappa,
        "p_value": None,
        "ci_lower": None,
        "ci_upper": None,
    })

stat_df = pd.DataFrame(stat_rows)
stat_path = MODELS_DIR / "statistical_tests.csv"
stat_df.to_csv(stat_path, index=False)
logger.info("Statistical tests saved to %s", stat_path)

## 16. Upload Artifacts to Scaleway

In [38]:
if COLAB:
    import glob
    model_dir = "../data/models"
    uploads = []
    for pattern in ["*.pth", "*.pkl", "*.csv", "*.json"]:
        uploads.extend(glob.glob(os.path.join(model_dir, pattern)))
    for filepath in sorted(uploads):
        fname = os.path.basename(filepath)
        s3.upload_file(filepath, ARTIFACT_BUCKET, f"models/{fname}")
        print(f"Uploaded {fname}")
else:
    subprocess.run(
        ["uv", "run", "python", "-m", "dronedetect.upload", "--all"],
        cwd="..", check=True,
    )

Uploaded confusion_matrices.json
Uploaded model_comparison_full.pkl
Uploaded model_comparison_results.csv
Uploaded per_class_metrics.csv
Uploaded resnet50_cnn.pth
Uploaded rfuavnet_iq.pth
Uploaded robustness_metrics.csv
Uploaded statistical_tests.csv
Uploaded svm_psd_drone.pkl
Uploaded vgg16_cnn.pth
